data ingestion to vector db pipeline


In [183]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [184]:
def process_all_pdfs(pdf_directory):
    """Process all pdf files in a directory"""
    all_documents=[]
    pdf_dir=Path(pdf_directory)

    #Find all pdf files recursively
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to Process")

    for pdf_file in pdf_files:
        print(f"\n Processing: {pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()

            #Add source information to metadata
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f" ERROR : {e}")
    print(f"\n Total documents loaded: {len(all_documents)}")
    return all_documents





In [185]:
all_pdf_documents=process_all_pdfs("../data")

Found 3 PDF files to Process

 Processing: attention_mechanism.pdf
Loaded 44 pages

 Processing: embeddings.pdf
Loaded 27 pages

 Processing: object_detection.pdf
Loaded 88 pages

 Total documents loaded: 159


In [186]:
all_pdf_documents

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'author': 'Hasi Hays', 'doi': 'https://doi.org/10.48550/arXiv.2601.03329', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Attention mechanisms in neural networks', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2601.03329v1', 'source': '..\\data\\pdf\\attention_mechanism.pdf', 'total_pages': 44, 'page': 0, 'page_label': 'i', 'source_file': 'attention_mechanism.pdf', 'file_type': 'pdf'}, page_content='Q\nKV\nsoftmax\nα ij\nAttention Mechanisms\nin Neural Networks\nA Comprehensive Mathematical Treatment\nFrom Theory to Implementation\nHasi Hays\narXiv:2601.03329v1  [cs.LG]  6 Jan 2026'),
 Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'author': 'Hasi Hays', 'doi': 'https://

In [187]:
###Text splitting get into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    ##Split documents into smaller cuncks for better RAG performance
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )

    split_docs=text_splitter.split_documents(documents)
    print(f"Split {documents} documents into {len(split_docs)} chuncks")

    #Show example of chunck

    if split_docs:
        print(f"\n Example chunck:")
        print(f"Content: {split_docs[0].page_content[:200]}......")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

In [188]:
chunks=split_documents(all_pdf_documents)
chunks

Split [Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'author': 'Hasi Hays', 'doi': 'https://doi.org/10.48550/arXiv.2601.03329', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Attention mechanisms in neural networks', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2601.03329v1', 'source': '..\\data\\pdf\\attention_mechanism.pdf', 'total_pages': 44, 'page': 0, 'page_label': 'i', 'source_file': 'attention_mechanism.pdf', 'file_type': 'pdf'}, page_content='Q\nKV\nsoftmax\nα ij\nAttention Mechanisms\nin Neural Networks\nA Comprehensive Mathematical Treatment\nFrom Theory to Implementation\nHasi Hays\narXiv:2601.03329v1  [cs.LG]  6 Jan 2026'), Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'author': 'Hasi Hays', 'doi': 'htt

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'author': 'Hasi Hays', 'doi': 'https://doi.org/10.48550/arXiv.2601.03329', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Attention mechanisms in neural networks', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2601.03329v1', 'source': '..\\data\\pdf\\attention_mechanism.pdf', 'total_pages': 44, 'page': 0, 'page_label': 'i', 'source_file': 'attention_mechanism.pdf', 'file_type': 'pdf'}, page_content='Q\nKV\nsoftmax\nα ij\nAttention Mechanisms\nin Neural Networks\nA Comprehensive Mathematical Treatment\nFrom Theory to Implementation\nHasi Hays\narXiv:2601.03329v1  [cs.LG]  6 Jan 2026'),
 Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:57610bf)', 'creationdate': '', 'author': 'Hasi Hays', 'doi': 'https://

embeddings and vector db


In [189]:
import numpy as np
from sentence_transformers  import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [190]:
class EmbeddingManager:
    """Handels documents embedding generation using SentenceTransformer"""
    def __init__(self,model_name:str ="all-MiniLM-L6-v2"):
        """
        Initialize the embeddig manager

        Args:
            model_name: HugginFace model name for sentence embeddigs
        """
        self.model_name=model_name
        self.model=None
        self._load_model()



    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embeddigs model:{self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully.Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model.{self.model_name}:{e}")
            raise

    def generate_embeddigs(self,texts:List[str]) ->np.array:
        """  
        Generate embeddigs for list of texts

        Arg:
            texts:List of texts strings to embed


        returns:
            numpy array of embeddigs with shape (len(texts),embeddig_dim)
        """
        if not self.model:
            raise ValueError("Model not Loaded")

        print(f"Generating embeddigs for {len(texts)} texts .......")
        embeddings=self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddigs with shape: {embeddings.shape}")
        return embeddings
    

    # def get_embeddigs_dimension(self) -> int:
    #     """Get the embeddig dimention of the model"""
    #     if not self.model:
    #         raise ValueError("Model is not loaded")
    #     return self.model.get_embedding_dimension()


In [191]:
#initialize the embeddig manager
embedding_manager=EmbeddingManager()
embedding_manager

Loading embeddigs model:all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6105.59it/s]


Model loaded successfully.Embedding dimension: 384


C:\Users\tanvi\AppData\Local\Temp\ipykernel_20160\1148031027.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully.Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


VECTOR STORE

In [192]:
class VectorStore:
    """Mnages document embeddigs in a ChromaDB vector store"""

    def __init__(self,collection_name: str="pdf_documents", persist_directory: str="../data/vector_store"):
        """"    
        Initialize the vectore store

        Args:
        collections_name: Nmae of the ChromaDB collection
        persist_directory: Directory to persist the vector store
        
        
        """

        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()


    def _initialize_store(self):
        """Initialize ChromaDB client"""
        try:
            #Create persistent ChromaDB client
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embeddigs for RAG"}
            )

            print(f"vector store initiazed.Collection:{self.collection_name}")
            print(f"Existing documents in collection : {self.collection.count}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise 

    def add_documents(self,documents:List[Any], embeddings: np.array):
        """  
        Add documents and their embeddings to the vector store

        Ars:
            documents:List of lanchain documents
            embeddigs: Correspoding embeddings for the documents
        
        """

        if len(documents)!= len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding {len(documents)} documents to vector store")


        #Prepare data for ChromaDB
        ids=[]
        metadatas=[]
        documents_text=[]
        embedding_list=[]



        for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
            #generate unique id
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare metadata
            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            #prepare content
            documents_text.append(doc.page_content)

            #embedding
            embedding_list.append(embedding.tolist())

        #add to collection
        try:
                

            self.collection.add(
                ids=ids,
                embeddings=embedding_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collections: {self.collection.count()}")

        except Exception as e:
            print(f"ERROR adding documents to vector store: {e}")
            raise





In [193]:
vectorstore=VectorStore()
vectorstore

vector store initiazed.Collection:pdf_documents
Existing documents in collection : <bound method Collection.count of Collection(name=pdf_documents)>


In [194]:
### convert text to embeddings
texts=[doc.page_content for doc in chunks]
texts


['Q\nKV\nsoftmax\nα ij\nAttention Mechanisms\nin Neural Networks\nA Comprehensive Mathematical Treatment\nFrom Theory to Implementation\nHasi Hays\narXiv:2601.03329v1  [cs.LG]  6 Jan 2026',
 'Abstract\nAttention mechanisms represent a fundamental paradigm shift in neural network architectures, enabling\nmodels to selectively focus on relevant portions of input sequences through learned weighting functions.\nThis monograph provides a comprehensive and rigorous mathematical treatment of attention mechanisms,\nencompassing their theoretical foundations, computational properties, and practical implementations in\ncontemporary deep learning systems. We begin by establishing the mathematical framework for attention\noperations, defining the core components of queries, keys, and values, and analyzing various scoring functions\nincluding additive, multiplicative, and scaled dot-product attention. The permutation equivariance prop-\nerty of self-attention is proven, and its implications for pos

In [195]:
#generate the embeddings
embeddings=embedding_manager.generate_embeddigs(texts)
embeddings

Generating embeddigs for 333 texts .......


Batches: 100%|██████████| 11/11 [00:06<00:00,  1.77it/s]

Generated embeddigs with shape: (333, 384)


array([[-0.05197832, -0.03591518,  0.01883071, ...,  0.04999439,
        -0.07526914, -0.02613254],
       [-0.00647856, -0.06132169,  0.01666066, ...,  0.07679995,
        -0.08416149, -0.04033048],
       [ 0.00546926, -0.03546318,  0.03088552, ...,  0.08900224,
        -0.02383825, -0.00246306],
       ...,
       [ 0.02794621, -0.01545429, -0.00904817, ...,  0.08002689,
        -0.06943569, -0.03358031],
       [ 0.03457044,  0.04931227, -0.01817937, ...,  0.04929694,
        -0.06155244, -0.03199129],
       [ 0.0524622 , -0.03557286,  0.01147245, ...,  0.07519849,
        -0.07084297,  0.05800091]], shape=(333, 384), dtype=float32)

In [196]:
#store in the vector database
vectorstore.add_documents(chunks,embeddings)

Adding 333 documents to vector store
Successfully added 333 documents to vector store
Total documents in collections: 1665


RAG Retriever

In [197]:
class RAGRetriever:
    """ Handles query based retrieval from the vector store"""


    def __init__(self,vectorstore:VectorStore, embedding_manager: EmbeddingManager):
        """"  
        
        Initialize the retriever

        Args:
            vectorestore: vector store containing document embeddings
            embedging_manager: Manager foor generating query embeddingd
        """


        self.vectorstore=vectorstore
        self.embedding_manager=embedding_manager



    def retrieve(Self,query:str, top_k: int=5, score_threshold: float=0.0)->List[Dict[str,Any]]:
        """   
        Retrieve relevent documents for a query


        Args:
            query:the search query
            top_k: Number of top results too return
            score_thresolf: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        
        
        """ 

        print(f"Retrieving for query: '{query}'")
        print(f"Top k: {top_k}, Score threshold: {score_threshold}")


        #generate query embeddings
        query_embedding=embedding_manager.generate_embeddigs([query])[0]


        #Search in vector space

        try:
            results=vectorstore.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            #process results
            retrieved_docs=[]

            if results["documents"] and results['documents'][0]:
                documents=results["documents"][0]
                metadatas=results['metadatas'][0]
                distances=results["distances"][0]
                ids=results["ids"][0]

                for i, (doc_id,document, metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    similarity_score=1-distance

                    if similarity_score>=score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content' : document,
                            'metadata' : metadata,
                            'similarity_score' : similarity_score,
                            'distance' : distance,
                            'rank' : i+1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")

            else:
                print("No documents found")
            return retrieved_docs


        except Exception as e:
            print(f"ERROR during retrieval: {e}")
            return []



rag_retriever=RAGRetriever(VectorStore,EmbeddingManager)
rag_retriever


    



In [198]:
rag_retriever.retrieve("Hey what is attention mechanism?")

Retrieving for query: 'Hey what is attention mechanism?'
Top k: 5, Score threshold: 0.0
Generating embeddigs for 1 texts .......


Batches: 100%|██████████| 1/1 [00:00<00:00, 30.94it/s]

Generated embeddigs with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_40222062_62',
  'content': '18 CHAPTER 3. SELF-ATTENTION MECHANISMS\nExample 3.2(Positional Patterns).Some attention heads learn to attend to specific relative positions.\nFor instance, a head might attend primarily to the next position (A i,i+1 ≈1) or the previous position\n(Ai,i−1 ≈1). These patterns extract positional information.\nExample 3.3(Syntactic Patterns).In language models, certain heads learn attention patterns that follow\nsyntactic structure. For example:\n•Heads that attend from verbs to their subjects\n•Heads that attend from pronouns to their antecedents\n•Heads that attend from words to their modifiers\nThese patterns emerge without explicit syntactic supervision, suggesting that attention mechanisms\ndiscover grammatical structure through language modeling objectives.\nExample 3.4(Broadcast Patterns).Some heads develop attention patterns where certain positions (often\nthe first position or special tokens) receive attention from most other positions. Th

In [207]:
#rag pipeline with groq llm
from langchain_ollama import ChatOllama





In [208]:

##Initialize the griq LLM 
llm = ChatOllama(
    model="llama3.2",
    temperature=0.1,
)


def rag_simple(query,retriever,llm,top_k=3):
    #retrieve the text

    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""


    if not context:
        return "No relevent context found to answer the question"
    
    #generate the answer using GROQ LLM


    prompt=f"""Use the following context to answer the quetion concisely.
        Context: {context}

        Question: {query}

        Answer:
    
    
    """

    response=llm.invoke([prompt.format(complex=context,query=query)])
    return response.content






In [211]:
ans=rag_simple("Hey what is embeddings?",rag_retriever,llm)

Retrieving for query: 'Hey what is embeddings?'
Top k: 3, Score threshold: 0.0
Generating embeddigs for 1 texts .......


Batches: 100%|██████████| 1/1 [00:00<00:00, 46.44it/s]

Generated embeddigs with shape: (1, 384)
Retrieved 3 documents (after filtering)


In [212]:
print(f"Answer: {ans}")

Answer: Embeddings are multidimensional points that represent words as locations in a semantic space, derived from the distributions of word neighbors. They are vectors used to capture the meaning of words.
